# Anonymize an audio file

Give the model two things:

- a **target** — the audio you want to anonymize. Its *words* are preserved.
- a **reference** — a donor voice. Its *voice* is what you hear in the output.

The output says the same thing in the donor's voice, so the original speaker's identity is not recoverable from the audio.

**Before running:** you need the XTTS v2 checkpoints (~2 GB, once). Either run `anonymize download-model` in a terminal, or uncomment the download cell below.

See [ANONYMIZATION.md](../ANONYMIZATION.md) for the full documentation.

In [ ]:
# Run once if you do not have the checkpoints yet (~2 GB).
# from anonymizer.download import download_model
# download_model()

## 1. Create the anonymizer

Nothing is loaded yet — this is free. The checkpoints load on the first `anonymize` call and stay loaded, so keep this object around.

In [ ]:
from anonymizer import Anonymizer

anon = Anonymizer()      # add mode="refine", language="de", device="cpu", ... as needed
anon.config

## 2. Pick your files

Point these at your own audio. For the donor you can also pass a **directory** of wavs — the conditioning is averaged across them, which gives a more stable voice than a single clip.

In [ ]:
target = "../data/TARGET_5s.wav"   # the audio to anonymize
reference = "../data/REF.wav"       # the donor voice to hide behind

import IPython.display as ipd
print("target (original speaker):")
ipd.display(ipd.Audio(target))
print("reference (donor voice):")
ipd.display(ipd.Audio(reference))

## 3. Anonymize

The first call loads the model, so it takes a while; later calls are fast. The transcript is produced with Whisper — pass `text="..."` if you already have one, and `language="de"` (etc.) for non-English audio.

In [ ]:
result = anon.anonymize(target, reference=reference)

print("transcript:", result.text)
anon.play(result)

## 4. Save it

In [ ]:
anon.save(result, "anonymized.wav")

## 5. Did it work?

Four numbers. The words should survive (**low** WER, **high** BLEU) while the identity should not (**low** similarity to the original speaker, **high** to the donor).

The absolute similarity values are small even when this works well — what matters is that `reference_similarity` is greater than `target_similarity`.

In [ ]:
report = anon.score(result, target)

for name, value in report.to_dict().items():
    print(f"{name:>22}: {value:.3f}")

## Going further

**Higher quality** — slower, wants a GPU:

```python
result = anon.anonymize(target, reference=reference, mode="iterate")  # or "refine"
```

**A whole folder**, resumable, with a manifest CSV:

```python
from anonymizer.batch import anonymize_directory

summary = anonymize_directory(
    "recordings/", "anonymized/", reference=reference, anonymizer=anon, resume=True
)
print(summary["completed"], "files ->", summary["manifest"])
```

**From a terminal:**

```bash
anonymize run interview.wav --reference donor.wav -o out.wav --score
anonymize batch recordings/ --reference donor.wav -o anonymized/ --resume
```

> **Note:** this hides *who* is speaking, not *what* was said. The transcript is
> reproduced verbatim, so identifying words — names, addresses — pass through. See the
> Limitations section in [ANONYMIZATION.md](../ANONYMIZATION.md).